In [2]:
import os
import sys
import random
import functools
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RepeatedKFold
import warnings
warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '3407'
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

def set_all_random_seeds(seed=3407):
    np.random.seed(seed)
    random.seed(seed)
    print(f"已设置随机种子为: {seed}")

def print_data_shapes(X_train, y_train, X_test, y_test):
    print(f"训练集形状: {X_train.shape}")
    print(f"测试集形状: {X_test.shape}")

def objective(trial, X_train, y_train, cv_folds, seed=3407):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'random_state': seed,
        'n_jobs': 1,                         
        'verbosity': 0
    }
    model = xgb.XGBRegressor(**params)
    
    rkf = RepeatedKFold(n_splits=cv_folds, n_repeats=5, random_state=seed)
    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=rkf, scoring='neg_mean_squared_error',
        n_jobs=1
    )
    mse_scores = -cv_scores
    avg_mse = np.mean(mse_scores)
    mse_std = np.std(mse_scores)
    stability_penalty = 0.1 * mse_std / avg_mse if avg_mse > 0 else 0
    return avg_mse + stability_penalty

def optimize_hyperparameters(X_train, y_train, n_trials=100, cv_folds=10, seed=3407):
    print("开始Optuna超参数优化...")

    sampler = TPESampler(
        seed=seed,
        n_startup_trials=10, 
        group=False,
        warn_independent_sampling=False
    )

    study = optuna.create_study(
        direction='minimize',
        study_name=f'xgboost_regression_seed{seed}',
        storage=None,
        load_if_exists=False,
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10) 
    )
    
    objective_partial = functools.partial(
        objective,
        X_train=X_train,
        y_train=y_train,
        cv_folds=cv_folds,
        seed=seed
    )
    
    study.optimize(objective_partial, n_trials=n_trials, n_jobs=1, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"\n最佳超参数: {best_params}")
    print(f"最佳交叉验证MSE: {study.best_value:.4f}")
    
    best_params_with_seed = best_params.copy()
    best_params_with_seed['random_state'] = seed
    return best_params_with_seed, study

def train_best_model(X_train, y_train, best_params, seed=3407):
    final_params = best_params.copy()
    final_params.update({
        'random_state': seed,
        'n_jobs': 1,
        'tree_method': 'hist',
        'deterministic_histogram': True,
        'verbosity': 0
    })
    model = xgb.XGBRegressor(**final_params)
    model.fit(
        X_train, y_train,
        verbose=False
    )
    return model, (X_train, y_train)

def evaluate_model(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    mse_diff = abs(train_mse - test_mse) / train_mse * 100 if train_mse > 0 else 0
    r2_diff = abs(train_r2 - test_r2) * 100
    
    print("\n" + "="*50)
    print("模型性能评估")
    print("="*50)
    print(f"{'指标':<15} {'训练集':<15} {'测试集':<15} {'差异(%)':<10}")
    print("-"*50)
    print(f"{'MSE':<15} {train_mse:<15.4f} {test_mse:<15.4f} {mse_diff:<10.2f}")
    print(f"{'RMSE':<15} {np.sqrt(train_mse):<15.4f} {np.sqrt(test_mse):<15.4f} {'-':<10}")
    print(f"{'MAE':<15} {train_mae:<15.4f} {test_mae:<15.4f} {'-':<10}")
    print(f"{'R²':<15} {train_r2:<15.4f} {test_r2:<15.4f} {r2_diff:<10.2f}")
    print("="*50)
    
    metrics_dict = {
        'train_mse': train_mse, 'test_mse': test_mse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'train_r2': train_r2, 'test_r2': test_r2,
        'mse_diff_percent': mse_diff, 'r2_diff_percent': r2_diff
    }
    predictions = {'y_train_pred': y_train_pred, 'y_test_pred': y_test_pred}
    return metrics_dict, predictions

def save_model(model, filepath='best_xgboost_model.pkl'):
    import joblib
    joblib.dump(model, 'XGB_regression_model.pkl')
    print(f"模型已保存到: {filepath}")

def xgboost_optimization_pipeline(X_train, y_train, X_test, y_test, n_trials=100, cv_folds=5, seed=3407):
    set_all_random_seeds(seed)
    print_data_shapes(X_train, y_train, X_test, y_test)
    
    best_params, study = optimize_hyperparameters(
        X_train, y_train, n_trials, cv_folds, seed
    )
    model, train_data = train_best_model(X_train, y_train, best_params, seed)
    metrics, predictions = evaluate_model(model, X_train, y_train, X_test, y_test)
    save_model(model, 'XGB_regression_model.pkl')
    print("\n优化完成！")
    return model, best_params, metrics, study, predictions, train_data

def main():
    SEED = 36
    
    X_train = pd.read_csv("x_train.csv")
    X_test  = pd.read_csv("x_test.csv")
    y_train = pd.read_csv("y_train.csv")
    y_test  = pd.read_csv("y_test.csv")
    
    if len(y_train.shape) > 1 and y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    if len(y_test.shape) > 1 and y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]
    
    model, best_params, metrics, study, predictions, train_data = xgboost_optimization_pipeline(
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        n_trials=100,
        cv_folds=10,
        seed=SEED
    )
    
    print(f"\n最佳参数数量: {len(best_params)}")
    print(f"训练集R²: {metrics['train_r2']:.4f}")
    print(f"测试集R²: {metrics['test_r2']:.4f}")
    
    results = {
        'model': model, 'best_params': best_params, 'metrics': metrics,
        'study': study, 'predictions': predictions, 'train_data': train_data
    }
    return results

if __name__ == "__main__":
    results = main()

[I 2026-04-08 11:17:29,589] A new study created in memory with name: xgboost_regression_seed36


已设置随机种子为: 36
训练集形状: (921, 30)
测试集形状: (231, 30)
开始Optuna超参数优化...


  0%|          | 0/100 [00:00<?, ?it/s]

Best trial: 0. Best value: 16.0578:   1%|          | 1/100 [00:23<38:39, 23.43s/it]

[I 2026-04-08 11:17:53,022] Trial 0 finished with value: 16.057797901662354 and parameters: {'n_estimators': 756, 'max_depth': 7, 'learning_rate': 0.25496836683723545, 'subsample': 0.5715867238605152, 'colsample_bytree': 0.9036795649350824, 'min_child_weight': 4, 'reg_alpha': 4.775417104143057e-07, 'reg_lambda': 2.1496172800756097e-06, 'gamma': 0.0010916885137868578}. Best is trial 0 with value: 16.057797901662354.


...

[I 2026-04-08 11:44:54,796] Trial 97 finished with value: 11.502979366493006 and parameters: {'n_estimators': 811, 'max_depth': 6, 'learning_rate': 0.013049276379127795, 'subsample': 0.5945486924296787, 'colsample_bytree': 0.5958857887522091, 'min_child_weight': 2, 'reg_alpha': 2.717180345987592, 'reg_lambda': 0.001026104167674978, 'gamma': 0.006280538037032239}. Best is trial 90 with value: 11.441763787899854.


Best trial: 90. Best value: 11.4418:  99%|█████████▉| 99/100 [27:42<00:18, 18.71s/it]

[I 2026-04-08 11:45:12,431] Trial 98 finished with value: 11.465525439608655 and parameters: {'n_estimators': 818, 'max_depth': 6, 'learning_rate': 0.012881676115851075, 'subsample': 0.5971341535245347, 'colsample_bytree': 0.5948652614864387, 'min_child_weight': 2, 'reg_alpha': 0.761519824833592, 'reg_lambda': 0.0003281192348543335, 'gamma': 0.0037424815512619156}. Best is trial 90 with value: 11.441763787899854.


Best trial: 90. Best value: 11.4418: 100%|██████████| 100/100 [27:56<00:00, 16.77s/it]


[I 2026-04-08 11:45:26,095] Trial 99 finished with value: 11.787279379266987 and parameters: {'n_estimators': 819, 'max_depth': 5, 'learning_rate': 0.012799281544761152, 'subsample': 0.6228778291670672, 'colsample_bytree': 0.5921810247364606, 'min_child_weight': 2, 'reg_alpha': 0.6107890926916434, 'reg_lambda': 0.0036346234274749344, 'gamma': 0.004054821054459496}. Best is trial 90 with value: 11.441763787899854.

最佳超参数: {'n_estimators': 779, 'max_depth': 7, 'learning_rate': 0.014754867610563745, 'subsample': 0.5929644451951669, 'colsample_bytree': 0.5698497167251237, 'min_child_weight': 2, 'reg_alpha': 2.964025082151408, 'reg_lambda': 0.00020650917505555836, 'gamma': 0.0029941925865262504}
最佳交叉验证MSE: 11.4418

模型性能评估
指标              训练集             测试集             差异(%)     
--------------------------------------------------
MSE             2.6238          16.5262         529.85    
RMSE            1.6198          4.0652          -         
MAE             1.1152          2.5553       

In [3]:
print("\n" + "="*60)
print("结果摘要")
print("="*60)
print(f"最佳参数: {results['best_params']}")
print(f"测试集MSE: {results['metrics']['test_mse']:.4f}")
print(f"测试集R²: {results['metrics']['test_r2']:.4f}")
print("="*60)


结果摘要
最佳参数: {'n_estimators': 779, 'max_depth': 7, 'learning_rate': 0.014754867610563745, 'subsample': 0.5929644451951669, 'colsample_bytree': 0.5698497167251237, 'min_child_weight': 2, 'reg_alpha': 2.964025082151408, 'reg_lambda': 0.00020650917505555836, 'gamma': 0.0029941925865262504, 'random_state': 36}
测试集MSE: 16.5262
测试集R²: 0.7948
